In [1]:
# ===== CELL 1 — Imports, seed, device, config (note: 50 epochs, F import) =====
import time, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:", device)

QUICK_TEST = False
EPOCHS = 2 if QUICK_TEST else 50
BATCH_SIZE = 128
CIFAR_CLASSES = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']

Device: cuda


In [2]:
# ===== CELL 2 — CIFAR-10 data: 45k/5k/10k split + loaders =====
MEAN, STD = (0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)
train_tf = T.Compose([T.RandomCrop(32,padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(MEAN,STD)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize(MEAN,STD)])

full_aug   = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=train_tf)
full_clean = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=eval_tf)
test_set   = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=eval_tf)

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(full_aug), generator=g).tolist()
train_set = Subset(full_aug, perm[:45000])
val_set   = Subset(full_clean, perm[45000:])

train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   256,        shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  256,        shuffle=False, num_workers=2, pin_memory=True)
print(f"train={len(train_set)} val={len(val_set)} test={len(test_set)}")

100%|██████████| 170M/170M [00:05<00:00, 33.5MB/s]


train=45000 val=5000 test=10000


In [3]:
# ===== CELL 3 — Utilities: param count, train, eval, plots, filters =====
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval(); correct=total=0; loss_sum=0.0
    for x,y in loader:
        x,y=x.to(device),y.to(device); out=model(x)
        if criterion is not None: loss_sum+=criterion(out,y).item()*x.size(0)
        correct+=(out.argmax(1)==y).sum().item(); total+=x.size(0)
    return correct/total, (loss_sum/total if criterion is not None else None)

def train_model(model, label, epochs=EPOCHS, lr=0.01, weight_decay=5e-4):
    set_seed(); model=model.to(device)
    crit=nn.CrossEntropyLoss()
    opt=torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay, nesterov=True)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    h={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[],'epoch_time':[],'label':label,'params':count_params(model)}
    print(f"\n=== {label} | params={h['params']:,} | epochs={epochs} | lr={lr} ===")
    for ep in range(epochs):
        model.train(); t0=time.time(); rl=rc=rt=0
        for x,y in train_loader:
            x,y=x.to(device),y.to(device); opt.zero_grad()
            out=model(x); loss=crit(out,y); loss.backward(); opt.step()
            rl+=loss.item()*x.size(0); rc+=(out.argmax(1)==y).sum().item(); rt+=x.size(0)
        sch.step(); dt=time.time()-t0
        tr_loss,tr_acc=rl/rt, rc/rt
        va,vl=evaluate(model,val_loader,crit)
        h['train_loss'].append(tr_loss); h['val_loss'].append(vl)
        h['train_acc'].append(tr_acc);   h['val_acc'].append(va); h['epoch_time'].append(dt)
        print(f"  ep {ep+1:02d}/{epochs} train_loss={tr_loss:.3f} acc={tr_acc:.3f} | val_loss={vl:.3f} acc={va:.3f} | {dt:.1f}s")
    h['avg_epoch_time']=float(np.mean(h['epoch_time'])); return model,h

def plot_history(hs, title="Training curves"):
    fig,ax=plt.subplots(1,2,figsize=(13,4.5))
    for h in hs:
        ep=range(1,len(h['train_loss'])+1)
        ax[0].plot(ep,h['train_loss'],'--',label=f"{h['label']} train")
        ax[0].plot(ep,h['val_loss'],'-',label=f"{h['label']} val")
        ax[1].plot(ep,h['val_acc'],'-',label=f"{h['label']} val acc")
    ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].set_title(f'{title}: loss'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy'); ax[1].set_title(f'{title}: val acc'); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

def plot_confusion(model, label=""):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for x,y in test_loader:
            ps.append(model(x.to(device)).argmax(1).cpu()); ys.append(y)
    cm=confusion_matrix(torch.cat(ys).numpy(), torch.cat(ps).numpy())
    fig,axc=plt.subplots(figsize=(7,6))
    ConfusionMatrixDisplay(cm,display_labels=CIFAR_CLASSES).plot(ax=axc,xticks_rotation=45,colorbar=False)
    axc.set_title(f'Confusion matrix — {label}'); plt.tight_layout(); plt.show(); return cm

def show_first_conv_filters(model, label=""):
    fc=None
    for m in model.modules():
        if isinstance(m,nn.Conv2d): fc=m; break
    if fc is None or fc.in_channels!=3: print("No 3-ch first conv."); return
    w=fc.weight.detach().cpu(); w=(w-w.min())/(w.max()-w.min()+1e-8)
    grid=torchvision.utils.make_grid(w[:min(64,w.size(0))],nrow=8,padding=1)
    plt.figure(figsize=(6,6)); plt.imshow(grid.permute(1,2,0)); plt.axis('off')
    plt.title(f'First conv filters — {label}'); plt.show()

def report_test(model,label):
    acc,_=evaluate(model,test_loader); print(f"{label}: TEST accuracy = {acc*100:.2f}%"); return acc

In [4]:
# ===== CELL 4 — ResNet building blocks + ResNet-11/18/34 =====
class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1=nn.Conv2d(in_c,out_c,3,stride=stride,padding=1,bias=False); self.bn1=nn.BatchNorm2d(out_c)
        self.conv2=nn.Conv2d(out_c,out_c,3,stride=1,padding=1,bias=False);      self.bn2=nn.BatchNorm2d(out_c)
        self.shortcut=nn.Sequential()
        if stride!=1 or in_c!=out_c:
            self.shortcut=nn.Sequential(nn.Conv2d(in_c,out_c,1,stride=stride,bias=False), nn.BatchNorm2d(out_c))
    def forward(self,x):
        out=F.relu(self.bn1(self.conv1(x))); out=self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResNet(nn.Module):
    def __init__(self, num_blocks, num_classes=10, dropout=0.0):
        super().__init__(); self.in_c=64
        self.stem=nn.Sequential(nn.Conv2d(3,64,3,1,1,bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.layer1=self._stage(64, num_blocks[0],1); self.layer2=self._stage(128,num_blocks[1],2)
        self.layer3=self._stage(256,num_blocks[2],2); self.layer4=self._stage(512,num_blocks[3],2)
        self.pool=nn.AdaptiveAvgPool2d(1); self.dropout=nn.Dropout(dropout); self.fc=nn.Linear(512,num_classes)
    def _stage(self, out_c, n, stride):
        layers=[]
        for s in [stride]+[1]*(n-1):
            layers.append(BasicBlock(self.in_c,out_c,s)); self.in_c=out_c
        return nn.Sequential(*layers)
    def forward(self,x):
        x=self.stem(x); x=self.layer1(x); x=self.layer2(x); x=self.layer3(x); x=self.layer4(x)
        x=self.pool(x).flatten(1); x=self.dropout(x); return self.fc(x)

def resnet11(dropout=0.0): return ResNet([1,1,1,2], dropout=dropout)
def resnet18(dropout=0.0): return ResNet([2,2,2,2], dropout=dropout)
def resnet34(dropout=0.0): return ResNet([3,4,6,3], dropout=dropout)
print(f"ResNet-11 params: {count_params(resnet11()):,}")
print(f"ResNet-18 params: {count_params(resnet18()):,}")

ResNet-11 params: 9,623,882
ResNet-18 params: 11,173,962


In [ ]:
# ===== CELL 5 — Part A: train ResNet-11 (baseline) and ResNet-18 =====
r11,h_r11 = train_model(resnet11(), "ResNet-11", lr=0.1)
r18,h_r18 = train_model(resnet18(), "ResNet-18", lr=0.1)
plot_history([h_r11,h_r18], "ResNet-11 vs ResNet-18")
acc_r11=report_test(r11,"ResNet-11"); acc_r18=report_test(r18,"ResNet-18")
plot_confusion(r18, "ResNet-18")
print(f"\n{'Model':12s}{'Params':>12s}{'TestAcc':>10s}{'s/epoch':>10s}")
for h,acc in [(h_r11,acc_r11),(h_r18,acc_r18)]:
    print(f"{h['label']:12s}{h['params']:>12,}{acc*100:>9.2f}%{h['avg_epoch_time']:>9.1f}s")


=== ResNet-11 | params=9,623,882 | epochs=50 | lr=0.1 ===
  ep 01/50 train_loss=2.156 acc=0.265 | val_loss=1.902 acc=0.302 | 26.9s
  ep 02/50 train_loss=1.487 acc=0.452 | val_loss=1.330 acc=0.513 | 27.0s
  ep 03/50 train_loss=1.212 acc=0.560 | val_loss=1.085 acc=0.615 | 28.2s
  ep 04/50 train_loss=1.017 acc=0.638 | val_loss=0.949 acc=0.659 | 30.4s
  ep 05/50 train_loss=0.886 acc=0.687 | val_loss=1.050 acc=0.652 | 27.4s
  ep 06/50 train_loss=0.786 acc=0.724 | val_loss=1.017 acc=0.653 | 28.1s
  ep 07/50 train_loss=0.694 acc=0.758 | val_loss=0.816 acc=0.722 | 28.6s
  ep 08/50 train_loss=0.630 acc=0.781 | val_loss=0.624 acc=0.778 | 28.2s
  ep 09/50 train_loss=0.578 acc=0.798 | val_loss=0.579 acc=0.802 | 27.8s
  ep 10/50 train_loss=0.544 acc=0.812 | val_loss=0.686 acc=0.762 | 28.2s
  ep 11/50 train_loss=0.524 acc=0.818 | val_loss=0.665 acc=0.779 | 27.9s
  ep 12/50 train_loss=0.496 acc=0.830 | val_loss=0.584 acc=0.800 | 27.8s
  ep 13/50 train_loss=0.474 acc=0.836 | val_loss=0.580 acc=0.803 

In [ ]:
# ===== CELL 6 — Part B: dropout (p=0.3, 0.5) on both ResNets =====
r11_d3,h11_3=train_model(resnet11(0.3),"ResNet-11 drop=0.3",lr=0.1)
r11_d5,h11_5=train_model(resnet11(0.5),"ResNet-11 drop=0.5",lr=0.1)
r18_d3,h18_3=train_model(resnet18(0.3),"ResNet-18 drop=0.3",lr=0.1)
r18_d5,h18_5=train_model(resnet18(0.5),"ResNet-18 drop=0.5",lr=0.1)
plot_history([h_r11,h11_3,h11_5], "ResNet-11 dropout")
plot_history([h_r18,h18_3,h18_5], "ResNet-18 dropout")
report_test(r11_d3,"ResNet-11 drop=0.3"); report_test(r11_d5,"ResNet-11 drop=0.5")
report_test(r18_d3,"ResNet-18 drop=0.3"); report_test(r18_d5,"ResNet-18 drop=0.5")

In [ ]:
# ===== CELL 7 — Final comparison across all four architectures =====
# Replace the two 0.0 values with your BEST AlexNet / VGG test accuracies from Problems 1 & 2:
best_alex_acc, best_alex_params = 0.0, 6_975_178
best_vgg_acc,  best_vgg_params  = 0.0, 9_231_114
finals = [("Best AlexNet", best_alex_params, best_alex_acc),
          ("Best VGG",     best_vgg_params,  best_vgg_acc),
          ("ResNet-11",    h_r11['params'],  acc_r11),
          ("ResNet-18",    h_r18['params'],  acc_r18)]
print(f"{'Model':14s}{'Params':>12s}{'TestAcc':>10s}")
for n,p,a in finals: print(f"{n:14s}{p:>12,}{a*100:>9.2f}%")
names=[f[0] for f in finals]; accs=[f[2]*100 for f in finals]
plt.figure(figsize=(8,5))
bars=plt.bar(names,accs,color=['#4C72B0','#55A868','#C44E52','#8172B3'])
for b,a in zip(bars,accs): plt.text(b.get_x()+b.get_width()/2,a+0.2,f"{a:.1f}%",ha='center')
plt.ylabel('Test accuracy (%)'); plt.title('CIFAR-10 test accuracy by best-of-family model')
plt.xticks(rotation=15); plt.ylim(0,100); plt.grid(axis='y',alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# ===== CELL 8 — Bonus (+4): ResNet-34 =====
r34,h_r34 = train_model(resnet34(), "ResNet-34", lr=0.1)
plot_history([h_r18,h_r34], "ResNet-18 vs ResNet-34")
report_test(r34,"ResNet-34")
print(f"ResNet-34 params: {count_params(resnet34()):,}")